# 05 - RAG simple

RAG significa Retrieval-Augmented Generation: buscar contexto relevante y luego generar una respuesta usando ese contexto.


## Paso 0: base de conocimiento

En un proyecto real esto podría venir de PDFs, notas, páginas web o una base de datos. Para el taller usamos una lista pequeña.


In [ ]:
import numpy as np
import ollama

MODEL = "llama3.2"

BASE_DE_CONOCIMIENTO = [
    "Horario del examen final de Álgebra: lunes y miércoles de 10:00 a 12:00, aula 301.",
    "El profesor de Cálculo Diferencial es el Dr. García, cubículo: edificio 2, piso 3.",
    "La fecha del examen final de Programación es el 15 de julio de 2025.",
    "Para inscribirse a materias usa el sistema de control escolar: control.universidad.mx",
    "El laboratorio de computación está abierto de 8:00 a 22:00, edificio 5, planta baja.",
    "Beca universitaria: plazo de inscripción hasta el 30 de junio de 2025.",
    "La biblioteca tiene horario extendido en época de exámenes: 8:00 a 24:00.",
    "El correo de soporte técnico es soporte@universidad.mx.",
    "Programación II exige haber aprobado Programación I con nota mínima de 6.",
    "El centro de estudiantes está en el edificio 1, primer piso, oficina 108.",
]


## Paso 1: funciones de embeddings y similitud

Estas funciones son iguales a las del ejemplo anterior. RAG reutiliza búsqueda semántica como primer paso.


In [ ]:
def obtener_embedding(texto):
    return ollama.embeddings(model=MODEL, prompt=texto)["embedding"]

def similitud_coseno(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))


## Paso 2: precomputar embeddings

Esta celda puede tardar unos segundos porque consulta al modelo para cada fragmento de la base.


In [ ]:
embeddings_base = [obtener_embedding(texto) for texto in BASE_DE_CONOCIMIENTO]
print(f"Embeddings creados: {len(embeddings_base)}")


## Paso 3: recuperar contexto relevante

Dada una pregunta, buscamos los fragmentos más parecidos por significado.


In [ ]:
def buscar_contexto(pregunta, top_k=3):
    embedding_pregunta = obtener_embedding(pregunta)
    resultados = []
    for texto, embedding in zip(BASE_DE_CONOCIMIENTO, embeddings_base):
        similitud = similitud_coseno(embedding_pregunta, embedding)
        resultados.append((similitud, texto))
    return sorted(resultados, reverse=True)[:top_k]

pregunta = "¿Cuándo es el examen de Programación?"
for similitud, texto in buscar_contexto(pregunta):
    print(f"[{similitud:.3f}] {texto}")


## Paso 4: generar usando solo el contexto

El prompt obliga al modelo a basarse en los fragmentos recuperados. Si la respuesta no está ahí, debe decirlo.


In [ ]:
def responder_con_rag(pregunta):
    resultados = buscar_contexto(pregunta)
    contexto = "\n".join(texto for _, texto in resultados)
    prompt = f"""Contesta la pregunta usando SOLO la información del contexto.
Si la información no está en el contexto, di "No tengo esa información".

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta:"""
    return ollama.generate(model=MODEL, prompt=prompt)["response"].strip()


## Paso 5: probar preguntas

La última pregunta no está en la base. Es un caso útil para revisar si el sistema se mantiene dentro del contexto.


In [ ]:
preguntas = [
    "¿Cuándo es el examen de Programación?",
    "¿Cómo me inscribo a materias?",
    "¿Dónde está el centro de estudiantes?",
    "¿Cuándo juega México?",
]

for pregunta in preguntas:
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {responder_con_rag(pregunta)}")
    print()
